In [1]:
# Import Packages
import numpy as np
import matplotlib.pyplot as plt

import qutip as q
from tqdm.notebook import tqdm

####################################################################
# Import utilities
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

from utilities.functions import cat, sqv, pssqv, sq_cat
from utilities.artificial_samples import sample_homodyne
from utilities.MLE_class import MLE
from utilities.plotting import plot_dm, plot_Wigner, plot_fidelities
from utilities.benchmarking import diff_bins, diff_samples

# Set quadrature range parameters
x_lim = (-5, 5)
x_points = 1000
x_vec = np.linspace(x_lim[0], x_lim[1], x_points)

cutoff = 50

In [2]:
# Even cat state
state = cat(cutoff, 0, 1.5)

n_angles = 4
n_samples = 1000
# Sample state (bin_data=False to get raw quadrature samples expected by MLE)
thetas, samples = sample_homodyne(state, n_angles=n_angles, n_samples_theta=n_samples, x_vec=x_vec, bin_data=False)

In [8]:
max_iter = 1000
fid_threshold = 1e-15
# Initial state vacuum
rho_0 = np.zeros((cutoff, cutoff))
rho_0[0, 0] = 1.0

# Benchmark how many bins are needed

In [9]:
# Run for different bins
N_bins = np.arange(5, 20, 1)

results_bins = diff_bins(n_bins_array=N_bins, xlims=x_lim, x_points=x_points, samples=samples,
                    max_iter=max_iter, threshold=fid_threshold, rho_init=rho_0)

In [10]:
for n_bins_value, result in zip(N_bins, results_bins):
    rho_est = result.rho_out
    n_iter = len(result.fidelities)
    fidelity = q.fidelity(q.Qobj(rho_est), state)**2
    print(f"Number of bins: {n_bins_value}, Fidelity: {fidelity}, Iterations: {n_iter}")

Number of bins: 5, Fidelity: 0.9526093580316574, Iterations: 196
Number of bins: 6, Fidelity: 0.9925982450826741, Iterations: 132
Number of bins: 7, Fidelity: 0.9724395692733383, Iterations: 266
Number of bins: 8, Fidelity: 0.9886624319565006, Iterations: 281
Number of bins: 9, Fidelity: 0.9970197901185263, Iterations: 47
Number of bins: 10, Fidelity: 0.9969124790098768, Iterations: 33
Number of bins: 11, Fidelity: 0.9904722688861136, Iterations: 30
Number of bins: 12, Fidelity: 0.995859925443795, Iterations: 37
Number of bins: 13, Fidelity: 0.9934389885415038, Iterations: 97
Number of bins: 14, Fidelity: 0.9960178754159406, Iterations: 27
Number of bins: 15, Fidelity: 0.9966242001102306, Iterations: 40
Number of bins: 16, Fidelity: 0.995639324154765, Iterations: 24
Number of bins: 17, Fidelity: 0.9953857789104668, Iterations: 19
Number of bins: 18, Fidelity: 0.9967752047395605, Iterations: 22
Number of bins: 19, Fidelity: 0.9954736345450619, Iterations: 24


# Benchmark for samples needed
- $n_\theta$: number of angles sampled
- $n_{samples}$: number of samples per angle

In [17]:
n_angles = [2,3,4,5,6,7,8,9,10]
n_samples = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500]
n_angles_samples = [(a, s) for a in n_angles for s in n_samples]

# Set binning
n_bins = 20

results_samples = diff_samples(n_angle_samples=n_angles_samples, state=state, x_vec=x_vec, 
                               n_bins=n_bins, max_iter=max_iter, threshold=fid_threshold, rho_init=rho_0)

In [21]:
for n_angle_sample, result in zip(n_angles_samples, results_samples):
    rho_est = result.rho_out
    n_iter = len(result.fidelities)
    fidelity = q.fidelity(q.Qobj(rho_est), state)**2
    print(fr'n_a={n_angle_sample[0]}, n_s={n_angle_sample[1]}, Fidelity: {fidelity}, Iterations: {n_iter}')

n_a=2, n_s=50, Fidelity: 0.8859731882914041, Iterations: 31
n_a=2, n_s=100, Fidelity: 0.94939876605755, Iterations: 52
n_a=2, n_s=150, Fidelity: 0.9707717373413167, Iterations: 20
n_a=2, n_s=200, Fidelity: 0.9750983259046797, Iterations: 37
n_a=2, n_s=250, Fidelity: 0.9757187984429718, Iterations: 57
n_a=2, n_s=300, Fidelity: 0.9952081521384941, Iterations: 31
n_a=2, n_s=350, Fidelity: 0.9868245948625423, Iterations: 69
n_a=2, n_s=400, Fidelity: 0.9881396545477821, Iterations: 28
n_a=2, n_s=450, Fidelity: 0.9916313272071294, Iterations: 42
n_a=2, n_s=500, Fidelity: 0.9846174164540858, Iterations: 66
n_a=3, n_s=50, Fidelity: 0.8631649515959761, Iterations: 86
n_a=3, n_s=100, Fidelity: 0.6723631761991536, Iterations: 603
n_a=3, n_s=150, Fidelity: 0.9341094952316126, Iterations: 114
n_a=3, n_s=200, Fidelity: 0.9563492231755709, Iterations: 114
n_a=3, n_s=250, Fidelity: 0.9863630629933053, Iterations: 50
n_a=3, n_s=300, Fidelity: 0.9863493304643909, Iterations: 73
n_a=3, n_s=350, Fidelity: